# Tutorial 01: Capability Discovery in A2A Systems

A deep, implementation-grounded tutorial for `A2A/01_capability_discovery`, focused on discovering agent identity and tool capabilities before routing and invocation.

## 1) Where We Are in the Journey

In `00_Agents`, you built stable reusable agent services with consistent endpoints (`/agent-info`, `/tools`, `/invoke`).
That step established the service contract layer but did not yet solve dynamic selection at runtime.
This tutorial exists to build a capability registry from live agents so orchestration can become data-driven instead of hardcoded.

## 2) What You Will Learn

- How to discover available agents and their metadata from live endpoints
- How to build a capability registry using `agent-info` + `tools` responses
- Why capability discovery is a control-plane function in agentic systems
- How implementation details in this folder affect reliability and evolvability
- How this step prepares the system for routing in the next tutorial

## 3) Why This Matters

In production A2A environments, agents can change, scale, or be replaced. Static, hardcoded mappings break quickly.
Capability discovery makes the controller adaptive: it asks the system what exists before deciding what to call.
This pattern mirrors MCP-style tool introspection and is foundational for robust multi-agent orchestration.

## 4) Core Concept Deep Dive

### Intuition

Capability discovery is the equivalent of building a real-time service catalog. Instead of assuming which agent handles which task, the controller gathers facts directly from the agents.

### System design perspective

This tutorial models a two-call discovery pattern per agent:

1. `GET /agent-info` for identity, description, and invoke endpoint
2. `GET /tools` for executable capability schema

The resulting registry is a normalized intermediate representation that decouples upstream reasoning from downstream service details.

### Trade-offs

- Benefits: dynamic adaptation, reduced hardcoding, easier scaling to many agents
- Costs: extra network calls, dependency on endpoint health, need for schema consistency
- Design tension: strict parsing catches contract drift early, but tolerant parsing can improve resilience

### Subtle implementation detail in this folder

`Untitled.ipynb` defines `discover_capabilities()` twice; the second definition overrides the first. This is a key notebook engineering behavior to understand while learning step-by-step.

## 5) Code Walkthrough (Only `A2A/01_capability_discovery`)

Files in this step:

- `README.md`: environment setup, service startup order, and quick API checks
- `Untitled.ipynb`: discovery logic and registry printing

Main code elements in `Untitled.ipynb`:

- `AGENTS` list: declares discovery targets (`http://localhost:8000`, `http://localhost:8001`)
- `discover_agents()`: fetches each agent's `/agent-info`
- `discover_capabilities()`: combines `/agent-info` and `/tools` into a unified registry
- `pprint(registry)`: makes the discovered capability graph human-readable

Dependency on previous step:

This folder assumes `00_Agents` services are running and exposing compatible metadata contracts.

In [1]:
# 6) Executable Cell: Setup for discovery targets
AGENTS = [
    "http://localhost:8000",  # math-agent
    "http://localhost:8001"   # finance-agent
]

print('Configured agents:', AGENTS)

Configured agents: ['http://localhost:8000', 'http://localhost:8001']


In [ ]:
# 7) Executable Cell: Light connectivity check (non-destructive)
import requests

def check_agents(agents):
    status = {}
    for url in agents:
        try:
            resp = requests.get(f"{url}/agent-info", timeout=2)
            if resp.status_code == 200:
                status[url] = "UP (200)"
            else:
                status[url] = f"DOWN (HTTP {resp.status_code})"
        except Exception as e:
            status[url] = f"DOWN ({type(e).__name__})"
    return status

health = check_agents(AGENTS)
health

{'http://localhost:8000': 'UP (404)',
 'http://localhost:8001': 'DOWN (ConnectionError)'}

In [ ]:
# 8) Executable Cell: discover_agents() from target notebook logic
def discover_agents():
    agents = []

    for url in AGENTS:
        info = requests.get(f"{url}/agent-info").json()
        agents.append(info)

    return agents

if all(v.startswith('UP') for v in health.values()):
    print(discover_agents())
else:
    print('Start agents first, then rerun this cell.')

In [ ]:
# 9) Executable Cell: first discover_capabilities() variant (tolerant parsing)
def discover_capabilities():
    registry = []

    for url in AGENTS:
        info = requests.get(f"{url}/agent-info").json()
        tools = requests.get(f"{url}/tools").json()

        registry.append({
            "agent": info.get("name", "unknown"),
            "description": info.get("description", "N/A"),
            "endpoint": info.get("endpoint", "N/A"),
            "tools": tools.get("tools", [])
        })

    return registry

if all(v.startswith('UP') for v in health.values()):
    from pprint import pprint
    pprint(discover_capabilities())
else:
    print('Agents unavailable; discovery skipped.')

In [ ]:
# 10) Executable Cell: second discover_capabilities() variant (strict parsing)
# This matches the redefinition pattern in Untitled.ipynb (overrides prior function).
def discover_capabilities():
    registry = []

    for url in AGENTS:
        info = requests.get(f"{url}/agent-info").json()
        tools = requests.get(f"{url}/tools").json()

        registry.append({
            "agent": info["name"],
            "description": info["description"],
            "endpoint": info["endpoint"],
            "tools": tools["tools"]
        })

    return registry

In [ ]:
# 11) Executable Cell: final registry output (as in target notebook)
if all(v.startswith('UP') for v in health.values()):
    registry = discover_capabilities()
    from pprint import pprint
    pprint(registry)
else:
    print('Start the target folder agents and rerun from Cell 7.')

## 6) System Flow Explanation

1. Controller reads configured agent base URLs from `AGENTS`.
2. For each URL, controller fetches `/agent-info` for identity + invocation endpoint.
3. Controller fetches `/tools` for executable capability schema.
4. Controller normalizes both responses into a single registry entry per agent.
5. Registry becomes the control-plane map used by later routing and invocation steps.
6. Runtime behavior depends on endpoint health and metadata contract consistency.

## 7) Important Concepts You Might Miss

- Discovery is a control-plane operation, not business logic execution.
- The registry structure is an interface boundary for downstream routing layers.
- Two function definitions with the same name in a notebook result in last-definition-wins semantics.
- Tolerant vs strict parsing is a reliability decision, not just coding style.
- This step assumes stable contracts from `00_Agents`; weak contracts propagate fragility forward.

## 8) Common Mistakes / Pitfalls

- Agent services are not running, causing connection errors during discovery.
- Ports in `AGENTS` do not match active service ports.
- Assuming `.json()` always returns expected keys when strict parsing is used.
- Forgetting that the second `discover_capabilities()` definition overrides the first.
- Treating discovered endpoint metadata as always trustworthy without validation.

## 9) Key Takeaways

- `A2A/01_capability_discovery` converts static assumptions into runtime capability knowledge.
- The registry output is the core artifact produced by this tutorial step.
- Discovery quality directly impacts routing quality in multi-agent systems.
- Notebook execution order matters because later function definitions can replace earlier ones.
- This folder operationalizes the contracts introduced in `00_Agents`.

## 10) What’s Next (Strict Preview)

Next, `02_ Agent Routing` tackles selection: deciding which discovered agent should handle a user query.
It solves the gap between capability registry data and an actionable routing decision.
This matters because discovery alone only describes the system; routing makes it operational.
You will move from "what agents can do" to "which agent should execute now."